In [ ]:
import pandas as pd
import numpy as np

# 📥 Cargar el archivo CSV
df = pd.read_csv("tabla_gamma.csv", sep=";", index_col=0)

# Extraer ángulos horizontales y verticales
horizontal_angles = df.columns.astype(float)
vertical_angles = df.index.astype(float)
gamma_matrix = df.to_numpy()

# 🔢 Interpolación bilineal
def interpolar_bilineal(x, y, x1, x2, y1, y2, Q11, Q21, Q12, Q22):
    if (x2 - x1) == 0 or (y2 - y1) == 0:
        return np.nan
    return (
        Q11 * (x2 - x) * (y2 - y) +
        Q21 * (x - x1) * (y2 - y) +
        Q12 * (x2 - x) * (y - y1) +
        Q22 * (x - x1) * (y - y1)
    ) / ((x2 - x1) * (y2 - y1))

# 🔍 Buscar gamma interpolado
def buscar_gamma_interpolado(h, v, h_angles, v_angles, gamma_matrix):
    h_angles = np.array(h_angles)
    v_angles = np.array(v_angles)

    # Si el valor está directamente en la tabla, usarlo sin interpolar
    if h in h_angles and v in v_angles:
      i = np.where(v_angles == v)[0][0]
      j = np.where(h_angles == h)[0][0]
      valor_directo = gamma_matrix[i, j]
      if not np.isnan(valor_directo):
          return f"Valor gamma exacto: {valor_directo:.2f}"
    # Verificar si el valor existe en la tabla con los ángulos invertidos
      if v in h_angles and h in v_angles:
          i = np.where(v_angles == h)[0][0]
          j = np.where(h_angles == v)[0][0]
          valor_invertido = gamma_matrix[i, j]
          if not np.isnan(valor_invertido):
              return f"Valor gamma exacto (por simetría): {valor_invertido:.2f}"


    def vecinos(valores, objetivo):
        menores = valores[valores < objetivo]
        mayores = valores[valores > objetivo]
        if len(menores) == 0 or len(mayores) == 0:
            return None
        return menores.max(), mayores.min()


    h1h2 = vecinos(h_angles, h)
    v1v2 = vecinos(v_angles, v)

    if h1h2 is None or v1v2 is None:
        return "Ángulo fuera del rango de la tabla."

    h1, h2 = h1h2
    v1, v2 = v1v2

    i1 = np.where(v_angles == v1)[0][0]
    i2 = np.where(v_angles == v2)[0][0]
    j1 = np.where(h_angles == h1)[0][0]
    j2 = np.where(h_angles == h2)[0][0]

    Q11 = gamma_matrix[i1, j1]  # (v1, h1)
    Q21 = gamma_matrix[i2, j1]  # (v2, h1)
    Q12 = gamma_matrix[i1, j2]  # (v1, h2)
    Q22 = gamma_matrix[i2, j2]  # (v2, h2)

# Interpolar Gamma invertido
    if np.isnan(Q11) or np.isnan(Q12) or np.isnan(Q21) or np.isnan(Q22):
    # Intentar invertir los ángulos
      h_inv, v_inv = v, h
      h1h2 = vecinos(h_angles, h_inv)
      v1v2 = vecinos(v_angles, v_inv)

      if h1h2 is None or v1v2 is None:
          return "No se puede interpolar: ángulos fuera del rango incluso tras invertir."

      h1, h2 = h1h2
      v1, v2 = v1v2

      i1 = np.where(v_angles == v1)[0][0]
      i2 = np.where(v_angles == v2)[0][0]
      j1 = np.where(h_angles == h1)[0][0]
      j2 = np.where(h_angles == h2)[0][0]

      Q11 = gamma_matrix[i1, j1]
      Q12 = gamma_matrix[i2, j1]
      Q21 = gamma_matrix[i1, j2]
      Q22 = gamma_matrix[i2, j2]

      if np.isnan(Q11) or np.isnan(Q12) or np.isnan(Q21) or np.isnan(Q22):
          return "No se puede interpolar: hay valores faltantes incluso tras invertir."

      gamma = interpolar_bilineal(h_inv, v_inv, h1, h2, v1, v2, Q11, Q21, Q12, Q22)
      return f"Valor gamma interpolado (con inversión): {gamma:.2f}"


# Interpolar Gamma directo
    gamma = interpolar_bilineal(h, v, h1, h2, v1, v2, Q11, Q12, Q21, Q22)
    return f"Valor gamma interpolado: {gamma:.2f}"

# 🧮 Consulta de múltiples pares
try:
    n = int(input("¿Cuántos pares de ángulos deseas ingresar? "))
except ValueError:
    print("Entrada inválida. Se usarán 10 por defecto.")
    n = 10

resultados = []

for i in range(n):
    print(f"\n🔢 Par {i+1}")
    try:
        h = float(input("Ingrese el ángulo horizontal: "))
        v = float(input("Ingrese el ángulo vertical: "))
        resultado = buscar_gamma_interpolado(h, v, horizontal_angles, vertical_angles, gamma_matrix)
        resultados.append((h, v, resultado))
    except ValueError:
        print("⚠️ Entrada inválida. Se omite este par.")
        resultados.append((None, None, "Entrada inválida"))

# 📋 Mostrar resultados
print("\n📊 Resultados:")
for idx, (h, v, res) in enumerate(resultados, start=1):
    print(f"{idx}. Ángulos: ({h}, {v}) → {res}")

# ➕ Calcular la sumatoria de los valores numéricos
suma_total = 0
for _, _, res in resultados:
    try:
        # Extraer el número desde el texto del resultado
        valor = float(res.split(":")[-1])
        suma_total += valor
    except:
        continue

print(f"\n🔢 Sumatoria total de valores gamma: {suma_total:.2f}")
